# Estimating fish locations with Pangeo-Fish

**Where might a fish have travelled between release and recapture?** In this Notebook, we combine temperatures recorded by a tag with ocean-model temperatures and acoustic detections to estimate possible locations over time.

We use tag `A19124` from the study by M. Gonse et al., "Combining acoustic telemetry with archival tagging to investigate the spatial dynamic of the understudied pollack, *Pollachius pollachius*", *Journal of Fish Biology* 106(5): 1400–1421 (2024), [doi:10.1111/jfb.15750](https://doi.org/10.1111/jfb.15750). Its Data Storage Tag (DST) records temperature and depth. Release and recapture/death information provide additional constraints.

Select the **Python (pangeo-fish)** kernel



**Workflow.**

1. **Prepare the analysis:** load the tools and choose the study settings.
2. **Compare temperatures:** compare tag observations with ocean-model temperatures at candidate locations.
3. **Use an equal-area grid:** represent candidate locations as HEALPix cells.
4. **Identify locations consistent with temperature:** give greater weight to closer temperature matches.
5. **Add acoustic detections:** constrain locations using receiver detections.
6. **Combine the evidence:** bring these observations together at each time step.
7. **Estimate movement:** fit how widely the fish can move between time steps.
8. **Estimate locations and tracks:** combine the observations with the movement model.
9. **Explore the results:** inspect maps, compare tracks, and make a video.


We use a Hidden Markov Model (HMM) to connect the observations over time. The fish's location is unknown; the model estimates its possible locations using both the observations and assumptions about movement.

## 1. Prepare the analysis

Run the cells from top to bottom. Click a code cell and press **Shift+Enter** to run it and move to the next cell. Wait for each calculation to finish before continuing. If a cell shows an error, ask an instructor before running the following steps. If you report a problem later, run the cell in the appendix **Information for bug reports** at the end of this notebook and copy its output into your report.

Run the setup cells below without changing them.

In [ ]:
# Use CPU execution.
import torch
import os

os.environ["CUDA_VISIBLE_DEVICES"] = ""

### Load the analysis tools

In [ ]:
from pint_xarray import unit_registry as ureg
import hvplot.xarray
import xarray as xr
import sys

# sys.path.append("../")
import pangeo_fish
from pangeo_fish.xarray_update import Accessor
from pangeo_fish.helpers import multiplot_healpix, multi_map_with_synced_sliders

### Study settings

All settings are collected here and grouped by the section that uses them. Keep the supplied values for your first run. For your own study, the tag, study area, temperature tolerance, and receiver detection range are particularly important choices.

In [ ]:
# SECTION 1 — Initialization and configuration
# Public tag identifier.
tag_name = "A19124"

# Tag data location: a public URL, local path, or s3:// bucket path.
tag_root = "https://data-taos.ifremer.fr/demo/tag/tag/"
tag_storage_options = None
# Backup: the same tag data mirrored on the Pangeo@EOSC MinIO (public, read-only).
# If data-taos.ifremer.fr is unreachable, use these two lines instead.
# tag_root = "s3://todaka-pangeo-fish-demo/tag"
# tag_storage_options = {
#     "anon": True,
#     "client_kwargs": {"endpoint_url": "https://pangeo-eosc-minioapi.vm.fedcloud.eu"},
# }

# Folder for saving results.
scratch_root = "./test"

# Connection settings for reading/writing results under scratch_root.
# None uses default settings.
# For an s3:// scratch_root, provide credentials/profile or endpoint options.
storage_options = None

# Number of time steps processed together. Default value.
# 1 = one time step per chunk; -1 = all time steps in one chunk.
chunk_time = 1

# SECTION 2 — Compare reference ocean data with DST observations
# Select the Copernicus Marine ocean model; keep only one assignment active.
# Both options work; better starting with IBI.

# IBI Reanalysis — Regional model, ~3 km, daily (default)
# https://data.marine.copernicus.eu/product/IBI_MULTIYEAR_PHY_005_002
copernicus_model_name = "cmems_mod_ibi_phy-temp_my_0.027deg_P1D-m"

# GLORYS12 Reanalysis — Global model, ~9 km, daily
# https://data.marine.copernicus.eu/product/GLOBAL_MULTIYEAR_PHY_001_030
# copernicus_model_name = "cmems_mod_glo_phy_my_0.083deg_P1D-m"

# Read the selected model from Copernicus Marine.
ref_url = None  # Default value

# Geographic bounds for model loading; also used in Sections 3 and 9.
bbox = {"latitude": [46, 51.5], "longitude": [-7, 0]}
# Reject cells where the model seabed is too shallow for the depths the fish
# was recorded at. A cell is rejected if its seabed is shallower than this
# fraction of the maximum tag depth in the same model time bin.
# Range: 0 or more; useful values are between 0 and 1.
#   0    = filter off: every cell is kept (default value)
#   0.8  = the seabed must be at least 80% of the deepest tag depth
#          (allows some error in the model bathymetry)
#   1    = the seabed must be at least as deep as the deepest tag depth
# Larger values reject more cells (stricter); values above 1 also reject
# cells whose seabed is only slightly deeper than the fish.
relative_depth_threshold = 0

# SECTION 3 — HEALPix regridding
# Use HEALPix cells for the analysis. Keep this setting for Sections 3–8.
dims = ["cells"]

# HEALPix resolution (approximate equivalent pixel size)
# Each refinement level divides the cell area by 4.
# Total number of cells = 12 * 4**refinement_level
# Equivalent size = sqrt(cell area) on a spherical Earth of radius 6371 km,
# not an exact geometric edge length.
#
# refinement_level = 9  # ~12.7 km
refinement_level = 10  # ~6.4 km
# refinement_level = 12  # ~1.6 km
#
# Higher refinement levels provide finer spatial resolution
# but increase the number of cells and computational cost.

# Earth shape used for the HEALPix regridding in Section 3. It is stored with
# the grid, and the maps and the movement search read it from there. Keep "sphere" here.
ellipsoid = "sphere"
# ellipsoid = "WGS84"  # Recommended when Earth's flattening matters for the required accuracy.

# SECTION 4 — Temperature-based emission probability
# Temperature tolerance (°C): larger values allow a wider range of
# differences between tag and model temperatures (Gaussian standard deviation).
differences_std = 0.75
# Spread around the known release and recapture/death locations: the standard
# deviation of a Gaussian, as an angle in radians on the Earth. Multiply by
# the Earth radius to get a distance: 1e-4 rad x 6371 km ≈ 0.64 km.
# This is smaller than one HEALPix cell (~6.4 km at level 10), so the location
# is effectively fixed to the release (or recapture) cell.
# Smaller values give tighter spatial constraints. Keep these default values.
initial_std = 1e-4
recapture_std = 1e-4

# SECTION 5 — Acoustic detection probability
# Maximum detection distance assumed around each receiver.
receiver_buffer_m = 2000

# SECTION 7 — Optimization of movement parameters
# The Earth radius used to convert movement distances into angles is derived
# from the ellipsoid stored with the HEALPix grid (about 6371 km for "sphere").
# Speed scale used to set the upper search limit for the movement parameter.
# This is not a strict speed limit on every possible movement.
maximum_speed_km_per_day = 60
# Multiplier for the upper limit of sigma values tried during fitting.
# 1 uses the limit set by maximum_speed and the model time interval;
# 2 doubles that limit. It does not directly multiply the fitted movement.
# Keep 1 by default. This is unrelated to video frame sampling.
adjustment_factor = 1
# Used with HealpixConv; not used by LargeHealpixConv or FoscatConv.
truncate = 4  # Default value
# Numerical precision of the parameter search. default value.
tolerance = 1e-6

# SECTION 8 — State probabilities and trajectories
# Trajectory summaries: mean position and most probable cell at each time step.
# Also used for plotting in Section 9.
track_modes = ["viterbi_healpix", "mode", "mean"]
# Supported here: "speed" (km/h) and "distance" to the previous point (km).
# Other quantities can be calculated afterward; see the optional note in Section 8.
additional_track_quantities = ["speed", "distance"]
# Manual-sigma comparisons use centroid/modal tracks; broad Viterbi neighborhoods
# can be very expensive. This is separate from the fitted model above.
comparison_track_modes = ["mode", "mean"]

# SECTION 9 — Visualization and video generation
# Number of data time steps between video frames
# 1 = include every time step
# 3 = include every third time step
# The current video tool omits the final time step; see Section 9.
time_step = 1

In [ ]:
# Compute dependent values after Papermill injects parameter overrides.
receiver_buffer = ureg.Quantity(receiver_buffer_m, "m")
maximum_speed = ureg.Quantity(maximum_speed_km_per_day, "km / day")

In [ ]:
# Define target root directories for storing analysis results.
target_root = f"{scratch_root}/{tag_name}"

# Defines default chunk size for optimization.
default_chunk = {"time": chunk_time, "lat": -1, "lon": -1}
default_chunk_dims = {"time": chunk_time}
default_chunk_dims.update({d: -1 for d in dims})

### Prepare the calculations

Run this cell to start Dask, which shares calculations across the available computing resources. No settings need to be changed here.

In [ ]:
# Set up a local cluster for distributed computing.
from distributed import LocalCluster

cluster = LocalCluster()
client = cluster.get_client()
client

### Load the tag observations

Load the temperature, depth, and event information for the selected fish.

In [ ]:
from pangeo_fish.helpers import load_tag

tag, tag_log, time_slice = load_tag(
    tag_root=tag_root, tag_name=tag_name, storage_options=tag_storage_options
)
tag

In [ ]:
# Legacy imported tags use Depth/Temp and negative depth.
# Public A19124 already has temperature and positive pressure.
if {"Depth", "Temp"}.issubset(tag["dst"].data_vars):
    dst = tag["dst"].to_dataset().rename({"Depth": "pressure", "Temp": "temperature"})
    dst["pressure"] = -dst[
        "pressure"
    ]  # inversion de signe sur la profondeur uniquement
    tag["dst"] = dst
    tag
    tag_log = tag_log.rename({"Depth": "pressure"})
    tag_log = tag_log.rename({"Temp": "temperature"})
    tag_log["pressure"] = -tag_log["pressure"]

Plot the recorded temperature and depth over time to inspect the observations before estimating locations.

In [ ]:
from pangeo_fish.helpers import plot_tag

plot = plot_tag(
    tag=tag,
    tag_log=tag_log,
    # you can directly save the plot if you want
    save_html=True,
    storage_options=storage_options,
    target_root=target_root,
)
plot

The red points are the observations used in the analysis, during the period when the tag is assumed to have been attached to a living fish. Check that this period agrees with the release and recapture/death information, and look for gaps in the records.

### Sign in to Copernicus Marine

Run the next cell. If prompted, enter your Copernicus Marine username and password. Your login is saved for later sessions; if it is already available, you will not be asked again.

If sign-in fails, ask an instructor or consult the [Copernicus Marine credentials guide](https://help.marine.copernicus.eu/en/articles/8185007-copernicus-marine-toolbox-credentials-configuration).

In [ ]:
from pathlib import Path

import copernicusmarine

credentials_file = Path.home() / ".copernicusmarine" / ".copernicusmarine-credentials"
if credentials_file.is_file():
    print("Using saved Copernicus Marine credentials.")
else:
    copernicusmarine.login()  # Prompts for username and password when needed.

## 2. Compare tag and ocean-model temperatures

### Load ocean temperatures for the study area

We use [IBI reanalysis](https://data.marine.copernicus.eu/product/IBI_MULTIYEAR_PHY_005_002/services), a regional ocean model with daily data and approximately 3 km resolution covering this study area. The following cell loads data for the selected region and tagging period.

[GLORYS12](https://data.marine.copernicus.eu/product/GLOBAL_MULTIYEAR_PHY_001_030/services) is a global alternative with daily data and approximately 9 km resolution. To switch models, add `#` at the start of the active `copernicus_model_name` line and remove it from the alternative line in the settings above. Use IBI for the first run.

In [ ]:
from pangeo_fish.helpers import load_model, compute_diff

reference_model = load_model(
    uri=ref_url,
    tag_log=tag_log,
    time_slice=time_slice,
    bbox=(bbox | {"max_depth": tag_log["pressure"].max()}),
    chunk_time=chunk_time,
    remote_options={},
    copernicus_model_name=copernicus_model_name,
)

### Compare temperatures at candidate locations

At each candidate location, compare the tag temperature with the model temperature at the observed depth. Observations are grouped into the model's time intervals. A difference closer to zero indicates a better temperature match, but several locations may have similar temperatures.

In [ ]:
%%time
diff = compute_diff(
    reference_model=reference_model,
    tag_log=tag_log,
    relative_depth_threshold=relative_depth_threshold,
    chunk_time=chunk_time,
)[0]
diff = diff.compute()
diff

### Check data availability over time

This plot counts the spatial cells with available data at each time step. Missing values, represented as `NaN` in Python, are excluded. This check is important before running the HMM.

Look for sudden drops or gaps. Counts can vary, but a sudden drop may indicate missing data or a processing problem. **If the count reaches zero, no spatial values are available at that time step: ask an instructor to help investigate before continuing.**

A temperature difference of zero is a perfect match, so `.count()` counts it as available data.

In [ ]:
%%time
diff["diff"].count(["lat", "lon"]).rename("Valid grid cells").plot()

In [ ]:
%%time
diff.to_zarr(
    f"{target_root}/diff.zarr", mode="w", storage_options=storage_options, zarr_format=2
)

## 3. Represent candidate locations on an equal-area grid

We transfer the temperature comparisons to a grid called **HEALPix**. Each cell represents the same area of the Earth's surface. Its hierarchical structure lets us choose a finer or coarser grid; the refinement level 10 uses an approximate cell size of **3.2 km**.

This cell size is not the accuracy of the estimated fish location. Location uncertainty also depends on the observations, the ocean model, and the movement assumptions. Equal area does not imply identical cell shapes or eliminate all spatial distortion; see the [HEALPix documentation](https://healpix.sourceforge.io/) for details.

In [ ]:
from pangeo_fish.helpers import (
    open_diff_dataset,
    regrid_dataset_hpresample,
)

In [ ]:
# Open the previous dataset (only necessary if you resume the notebook from here)
diff = open_diff_dataset(target_root=target_root, storage_options=storage_options)
diff

In [ ]:
reshaped, resampling_info = regrid_dataset_hpresample(
    ds=diff,
    refinement_level=refinement_level,
    ellipsoid=ellipsoid,
    dims=dims,
    mask_threshold=0.0,
)
reshaped

In [ ]:
resampling_info["bbox"] = bbox
import json

# Save the regridding metadata and geographic bounds alongside the results.
with open(f"{target_root}/resampling_info.json", "w") as f:
    json.dump(resampling_info, f)

with open(f"{target_root}/resampling_info.json") as f:
    resampling_info = json.load(f)
resampling_info

Check data availability again after changing the grid. The number of cells may differ because their size and coverage have changed. Look for sudden drops, and investigate any time step with a count of zero before continuing.

In [ ]:
reshaped["diff"].count(dims).rename("Valid HEALPix cells").plot()

In [ ]:
from pangeo_fish.storage import prepare_dggs_zarr

# Saves the result
prepare_dggs_zarr(reshaped.chunk(default_chunk_dims)).to_zarr(
    f"{target_root}/diff-regridded.zarr",
    mode="w",
    consolidated=True,
    compute=True,
    storage_options=storage_options,
    zarr_format=2,
)

The following maps show temperature differences (`diff`) and model seabed depth (`H0`). On the temperature-difference map, look for values close to zero: are good matches concentrated in one area or spread across several areas? Check how this pattern changes over time.

In [ ]:
reshaped.healpix.plot(var=["diff", "H0", "ocean_mask"], alpha=0.5)

## 4. Identify locations consistent with the temperatures

We give more weight to locations where model temperatures match the tag observations. These weights are called **emission likelihoods**: they describe how consistent the observations are with each candidate location.

We use a Gaussian curve centered on a temperature difference of zero. The setting `differences_std` controls its width: larger values allow a wider range of temperature differences. Release and recapture/death locations provide additional spatial constraints.

The temperature weights, stored as `emission_pdf["pdf"]`, do not yet sum to one across locations. We combine and normalize the evidence in Section 6.

In [ ]:
from pangeo_fish.helpers import compute_emission_pdf

In [ ]:
# Open the previous dataset (only necessary if you resume the notebook from here)
differences = xr.open_dataset(
    f"{target_root}/diff-regridded.zarr",
    engine="zarr",
    chunks={},
    storage_options=storage_options,
)

# ... and compute the emission matrices
emission_pdf = compute_emission_pdf(
    diff_ds=differences,
    events_ds=tag["tagging_events"].ds,
    differences_std=differences_std,
    initial_std=initial_std,
    recapture_std=recapture_std,
    dims=dims,
    chunk_time=chunk_time,
)[0]
emission_pdf

Compare the temperature weights (`pdf`) with the release (`initial`) and final-location (`final`, when available) maps. Do the temperature observations identify a small area, or leave several possible regions?

In [ ]:
emission_pdf.healpix.plot(["initial", "final", "pdf"])

Check the total temperature likelihood at each time step, `total = sum(dims, min_count=1)`. It need not equal one yet.

- `NaN`: no data at that time step.
- `0`: data exist, but no location matches the tag temperature.

If either appears, investigate it before continuing.

In [ ]:
emission_pdf = emission_pdf.chunk(default_chunk_dims).persist()
total = (
    emission_pdf["pdf"].sum(dims, min_count=1).rename("Total temperature likelihood")
)
print("Time steps without data (NaN):", int(total.isnull().sum()))
print("Time steps with zero total weight:", int((total == 0).sum()))
total.plot()

In [ ]:
from pangeo_fish.storage import prepare_dggs_zarr

# Save the dataset
prepare_dggs_zarr(emission_pdf).to_zarr(
    f"{target_root}/emission_diff.zarr",
    mode="w",
    consolidated=True,
    storage_options=storage_options,
    zarr_format=2,
)

In [ ]:
from pangeo_fish.storage import prepare_dggs_zarr

emission_pdf.attrs["sources"] = emission_pdf.attrs.get("sources", []) + [
    "temperature"
]  # Record the evidence sources included so far.
prepare_dggs_zarr(emission_pdf).to_zarr(
    f"{target_root}/total_pdf.zarr",
    mode="w",
    consolidated=True,
    storage_options=storage_options,
    zarr_format=2,
)

## 5. Add information from acoustic detections

A detection tells us that the fish was within the assumed detection area of a receiver. We use this information to constrain its possible locations, complementing the temperature evidence.

This step requires acoustic detections and receiver locations in the tag data. The setting `receiver_buffer` defines the assumed detection range. We combine this evidence with temperature in Section 6.

In [ ]:
from pangeo_fish.helpers import compute_acoustic_pdf

In [ ]:
# Load the previous emission pdf and compute the emission probabilities based on acoustic detections
emission_diff = xr.open_dataset(
    f"{target_root}/emission_diff.zarr",
    engine="zarr",
    chunks={},
    storage_options=storage_options,
)

acoustic_pdf = compute_acoustic_pdf(
    emission_ds=emission_diff,
    tag=tag,
    receiver_buffer=receiver_buffer,
    chunk_time=chunk_time,
    dims=dims,
)[0]
acoustic_pdf = acoustic_pdf.persist()
acoustic_pdf

In [ ]:
acoustic_pdf.healpix.plot("acoustic")

The following plot compares detection times with a cell-count diagnostic. Look for changes around detection times.

Here the code counts cells satisfying `acoustic != 0`. Missing values also satisfy this comparison, so this curve is not an exact count of possible fish locations. Keep this distinction in mind when interpreting it.

In [ ]:
tag["acoustic"]["deployment_id"].hvplot.scatter(c="red", marker="x") * (
    acoustic_pdf["acoustic"] != 0
).sum(dim=dims).hvplot()

At detection times, the possible locations can narrow sharply to the areas around receivers, under the assumed detection range. These detections provide strong spatial constraints. Interpret changes in the curve alongside the detection times, rather than treating its height as an exact number of possible locations.

The next cell optionally saves the acoustic results separately. The following analysis can use the results already calculated.

In [ ]:
from pangeo_fish.storage import prepare_dggs_zarr

prepare_dggs_zarr(acoustic_pdf).to_zarr(
    f"{target_root}/acoustic.zarr",
    mode="w",
    consolidated=True,
    storage_options=storage_options,
    zarr_format=2,
)

### Bring the evidence together

In [ ]:
# Open the currently saved evidence sources.
current_total = xr.open_dataset(
    f"{target_root}/total_pdf.zarr",
    engine="zarr",
    chunks={},
    storage_options=storage_options,
)
current_total

In [ ]:
from pangeo_fish.storage import prepare_dggs_zarr

if "acoustic" in current_total.data_vars:
    pass
else:
    current_total = current_total.load()
    current_total.attrs["sources"] = current_total.attrs.get("sources", []) + [
        "acoustic"
    ]

    merged_pdf = current_total.merge(
        acoustic_pdf.drop_indexes(["cell_ids"]), compat="override"
    )

    prepare_dggs_zarr(merged_pdf).to_zarr(
        f"{target_root}/total_pdf.zarr",
        mode="w",
        consolidated=True,
        storage_options=storage_options,
        zarr_format=2,
    )
merged_pdf

## 6. Add information from Bathymetry

In this step, the goal is to calculate another emission distribution, this time using a bathymetry like GEBCO, EMODNET...

These additional probabilities will enhance the emission pdf constructed in the previous step by incorporating information from bathymetry information.

/!\ you must have run and save at least once chapter 2 and the "relative_depth_threshold" parameter have to be put at 0

### Data importation

Import the ocean reference model and diff which was calculated in chapter 2. Also import of the bathymetry database.

In [ ]:
if "reference_model" in locals():
    pass
else:
    from pangeo_fish.helpers import load_model

    reference_model = load_model(
        uri=ref_url,
        tag_log=tag_log,
        time_slice=time_slice,
        bbox=(bbox | {"max_depth": tag_log["pressure"].max()}),
        chunk_time=chunk_time,
        remote_options={},
        copernicus_model_name=copernicus_model_name,
    )
emission_diff = xr.open_dataset(
    f"{target_root}/emission_diff.zarr",
    engine="zarr",
    chunks={},
    storage_options=storage_options,
)

In [ ]:
import fsspec
import numpy as np
from pangeo_fish.cf import bounds_to_bins
from pangeo_fish.tags import adapt_model_time, reshape_by_bins, to_time_slice
from pangeo_fish.bathy import batch_compute_pdf_bathy
from pangeo_fish.bathy import (
    compute_healpix_histogram_region_bin_size,
)

# bathy_url is the path to the bathymetry file
bathy_url = "https://data-taos.ifremer.fr/global_bathy/GEBCO_2024.zarr"

nside = (
    2**refinement_level
)  # parameter equivalent to refinement level used in some functions
full_bathy = xr.open_dataset(
    "https://data-taos.ifremer.fr/demo/database/gebco_2024_new.zarr/",
    engine="zarr",
    chunks={},
    storage_options=None,
).rename({"lat": "latitude", "lon": "longitude"})
full_bathy

In [ ]:
subset_bathy = full_bathy.sel(
    {dim: slice(bounds[0] - 0.1, bounds[1] + 0.1) for dim, bounds in bbox.items()}
)

In [ ]:
ds_histo = compute_healpix_histogram_region_bin_size(
    subset_bathy,
    nside=nside,
    max_depth_m=1000,  # <- max depth in meters
    depth_bin_size=16,  # <- bin width in meters
    ellipsoid=ellipsoid,
)

Compute the pdf

In [ ]:
from pangeo_fish.bathy import batch_compute_pdf_bathy

reshaped_tag = reshape_by_bins(  # reshape the tags for the calculation of the pdf
    tag_log,
    dim="time",
    bins=(
        reference_model.cf.add_bounds(["time"], output_dim="bounds")
        .pipe(bounds_to_bins, bounds_dim="bounds")
        .get("time_bins")
    ),
    other_dim="obs",
).chunk({"time": chunk_time})

pdf_da_func = batch_compute_pdf_bathy(
    ds_histo,
    reshaped_tag,
    target_root,
    batch_size=10000,
)

sum_over_cells = pdf_da_func.sum(dim="cells", skipna=True)
bathy_pdf = pdf_da_func / sum_over_cells  # normalization
bathy_pdf

The next cell optionally saves the bathymetric results separately. The following analysis can use the results already calculated.

In [ ]:
from pangeo_fish.storage import prepare_dggs_zarr

prepare_dggs_zarr(bathy_pdf.compute()).to_zarr(
    f"{target_root}/bathy_pdf_{tag_name}.zarr",
    compute=True,
    mode="w",
    consolidated=True,
    zarr_format=2,
    storage_options=storage_options,
)

In [ ]:
# If you want to see the pdf
bathy_pdf.healpix.plot("pdf_bathy", refinement_level=refinement_level)

### Bring the evidence together

In [ ]:
## Open the dataset with the other pdfs
current_total = xr.open_dataset(
    f"{target_root}/total_pdf.zarr",
    engine="zarr",
    chunks={},
    storage_options=storage_options,
)

In [ ]:
from pangeo_fish.storage import prepare_dggs_zarr

if "pdf_bathy" in current_total.data_vars:
    merged_pdf = current_total
    pass
else:
    current_total = current_total.load()
    current_total.attrs["sources"] = list(
        set(current_total.attrs.get("sources", [])) | {"bathymétrie"}
    )
    merged_pdf = current_total.merge(
        bathy_pdf.drop_indexes(["cell_ids"]), compat="override"
    )

    prepare_dggs_zarr(merged_pdf).to_zarr(
        f"{target_root}/total_pdf.zarr",
        mode="w",
        consolidated=True,
        storage_options=storage_options,
        zarr_format=2,
    )
merged_pdf

In [ ]:
liste = multiplot_healpix(
    [(merged_pdf, ["pdf", "pdf_bathy"])],
)

multi_map_with_synced_sliders(liste)

## 7. Add information from Tide
In this step, the goal is to calculate another emission distribution based on tide.

First we detect tide, then we try to observe a behavior on the data



In [ ]:
from pangeo_fish.tide import *
import xarray as xr
import pandas as pd
import numpy as np

In [ ]:
tag_test = detect_tide(tag_log, tag_name)
plot_tide(tag_test)
tag_test

In [ ]:
tag_test.to_zarr(
    f"{target_root}/tide_behav.zarr",
    mode="w",
    consolidated=True,
    storage_options=storage_options,
    zarr_format=2,
)

In [ ]:
tide_behav = xr.open_dataset(f"{target_root}/tide_behav.zarr", engine="zarr")
plot_tide(tide_behav)
tide_behav["behav_found"].plot()
plt.figure()
tide_behav["tide_found"].plot()

### Import Tide database

In [ ]:
import requests

url = "https://data-taos.ifremer.fr/demo/database/Subset_NS_EC_forTMD30_21_2_2025.nc"

# download the file
print("Téléchargement...")
r = requests.get(url)
with open("Subset_NS_EC_forTMD30_21_2_2025.nc", "wb") as f:
    f.write(r.content)

# Open with xarray
print("Opening...")
data_360 = xr.open_dataset("Subset_NS_EC_forTMD30_21_2_2025.nc")
data = convert_lon_360_to_180(
    data_360, bbox
)  # conversion in the good system of coordinates
data

In [ ]:
ds_pdf = tide_pdf(tide_behav, data)
ds_pdf = ds_pdf.assign_coords(longitude=((ds_pdf["longitude"] + 180) % 360) - 180)

# la grille doit toujours être triée pour que le regrid soit cohérent
ds_pdf = ds_pdf.sortby("longitude")

In [ ]:
# Save in zarr
ds_pdf.to_zarr(
    f"{target_root}/tide_pdf.zarr",
    mode="w",
    consolidated=True,
    storage_options=storage_options,
    zarr_format=2,
)

In [ ]:
ds_pdf

### Healpix regridding

In [ ]:
tide_pdf, tide_found_daily, behav_found_daily = tide_to_healpix(
    target_root, refinement_level, ellipsoid
)

In [ ]:
ds = tide_pdf.copy()

In [ ]:
neg_min = ds["pdf"].where(ds["pdf"] < 0).min(dim="time").compute()
import matplotlib.pyplot as plt

mask = neg_min.notnull()
plt.figure(figsize=(8, 4))
plt.scatter(
    ds["longitude"].where(mask), ds["latitude"].where(mask), c=neg_min.where(mask), s=3
)
plt.colorbar(label="minimum de pdf")
plt.xlabel("lon")
plt.ylabel("lat")
plt.show()

In [ ]:
da = ds["pdf"]

# Y a-t-il au moins une valeur négative ?
print(bool((da < 0).any().compute()))

# Combien de valeurs négatives ?
print(int((da < 0).sum().compute()))

# Minimum de la variable
print(float(da.min().compute()))

In [ ]:
tide_pdf.healpix.plot("pdf")

In [ ]:
tide_pdf

In [ ]:
behav_found_daily.to_zarr(
    f"{target_root}/behav_healpix.zarr",
    mode="w",
    consolidated=True,
    storage_options=storage_options,
    zarr_format=2,
)

### Merge with other pdfs

In [ ]:
emission_with_tide, tide_mask, behav_mask = merge_tide_with_total(
    target_root, storage_options
)

In [ ]:
from pangeo_fish.storage import prepare_dggs_zarr

prepare_dggs_zarr(emission_with_tide).to_zarr(
    f"{target_root}/total_pdf.zarr",
    mode="w",
    consolidated=True,
    storage_options=storage_options,
    zarr_format=2,
)
emission_with_tide

## 8. Combine the distributions

We multiply the different probabilty density functions and include the final-location information when available. We then divide by the total weight across cells so that the combined values sum to one at each time step. This is called **normalization**.

The result describes the relative support for candidate locations from the observations. We still need the movement model to connect locations over time.

In [ ]:
from pangeo_fish.helpers import normalize_pdf

In [ ]:
current_total = xr.open_dataset(
    f"{target_root}/total_pdf.zarr",
    engine="zarr",
    chunks={},
    storage_options=storage_options,
)
current_total

In [ ]:
from pangeo_fish.storage import prepare_dggs_zarr

emission_pdf = normalize_pdf(
    ds=current_total,
    chunks=default_chunk_dims,
    dims=dims,
    exclude=("initial", "final", "mask"),
    excluded_pdf=None,  # ("pdf",)
)[0]

prepare_dggs_zarr(emission_pdf).to_zarr(
    f"{target_root}/combined.zarr",
    mode="w",
    consolidated=True,
    storage_options=storage_options,
    zarr_format=2,
)
combined = emission_pdf

emission_pdf["pdf"].sum(dims).hvplot(
    width=500, ylim=(0, 2), title="Total probability per time step"
)

In [ ]:
variables_a_exclure = ["final", "initial", "mask"]

noms = [v for v in current_total.data_vars if v not in variables_a_exclure]
noms
liste = multiplot_healpix(
    [(current_total, noms)],
)

multi_map_with_synced_sliders(liste)

The total probability should be approximately **1** at every time step. A time step with no data also shows up as a departure from 1 (its sum is 0). Investigate any such time step before fitting the movement model.

In [ ]:
combined.healpix.plot("pdf")

## 9. Estimate how widely the fish can move

The movement model connects possible fish locations between successive time steps. Its parameter, `sigma`, controls how widely movement can spread: larger values allow a broader range of displacements. We estimate this parameter from the observations.

Run the following cells to fit and save the movement parameter. Check that the search finishes without warnings. The printed `f(x)` values measure the fit at trial values of `sigma`; they need not improve at every step while the search explores different values.

The movement time interval comes from the dates in the data. Changing the video setting `time_step` does not change fish movement in the model.

<details>
<summary>Optional: how the parameter search works</summary>

The search uses Brent's method between `1e-4` radians and an upper bound of `maximum_speed × Δt × adjustment_factor / earth_radius`, where Δt is the interval in the emission time coordinate and `earth_radius` is derived from the ellipsoid stored with the HEALPix grid (about 6371 km for `"sphere"`). The speed setting bounds the search for `sigma`; it is not a strict limit on every possible displacement.

Check whether the fitted value is near a search bound. Successful convergence does not establish that the solution is unique or the global optimum. The appendix allows comparison with alternative values. The fitted parameters and calculation settings are saved in a JSON file for the next step.

</details>

In [ ]:
from pangeo_fish.helpers import optimize_pdf

### Single sigma optimisation

This is the optimisation of the sigma for convolution without taking acount of any beahaviors. See base_demo.

### Multivariate optimisation

This optimisation use behavior to make sigma time-dependant. By doing so you can constraint the fish movement each days using behaviors detections.

here we use tidal information

In [ ]:
fit_root = f"{target_root}/two_sigma"

from pangeo_fish.helpers import (
    compute_detection_time_intervals,
    stamp_parameter_indices,
    stamp_parameter_indices_from_mask,
)
import numpy as np
import matplotlib.pyplot as plt

emission = xr.open_dataset(
    f"{target_root}/combined.zarr",
    engine="zarr",
    chunks=default_chunk_dims,
    inline_array=True,
    storage_options=storage_options,
)

behav_mask = xr.open_dataset(
    f"{target_root}/behav_healpix.zarr",
    engine="zarr",
    chunks=default_chunk_dims,
    inline_array=True,
    storage_options=storage_options,
)
behav_mask = (
    behav_mask["behav_found"]
    .isel(time=slice(1, len(behav_mask["behav_found"].values)))
    .values
)

behav_mask = (behav_mask > 0).astype(np.int8)
plt.plot(behav_mask)
pdf = stamp_parameter_indices_from_mask(
    pdf=emission,
    mask=behav_mask,
    sigma_bounds_dict={0: (0, maximum_speed), 1: (0, maximum_speed / 2)},
)

In [ ]:
from pangeo_fish.storage import prepare_dggs_zarr

from pangeo_fish.helpers import optimize_pdf

params, updated_emission = optimize_pdf(
    ds=pdf,
    adjustment_factor=adjustment_factor,
    truncate=truncate,
    maximum_speed=maximum_speed,
    tolerance=tolerance,
    dims=dims,
    single_behav=True,
    multivariate=True,
    # let's use another subfolder called "two_sigma"
    save_parameters=True,
    storage_options=storage_options,
    target_root=f"{fit_root}",
    conv_method="LargeHealpixConv",
)

updated_emission.attrs.pop("sigma_bounds", None)
prepare_dggs_zarr(updated_emission).to_zarr(
    f"{fit_root}/combined.zarr",
    mode="w",
    consolidated=True,
    storage_options=storage_options,
    zarr_format=2,
)
# Rendering reads the same grid metadata alongside the fitted model.
with open(f"{fit_root}/resampling_info.json", "w") as f:
    json.dump(resampling_info, f)
params

## 10. Estimate locations and summarize tracks

We combine the observational evidence with the fitted movement model to estimate a probability for each candidate location over time. We summarize these distributions in two ways:

- **`mean`:** the mean location at each time step.
- **`mode`:** the cell with the highest probability at each time step.

These lines summarize the estimated distributions; they are not directly observed paths. When interpreting them, also examine how widely the probability is spread across the map. We calculate distance between successive track points in kilometres and the corresponding speed in kilometres per hour.

<details>
<summary>Optional: other track methods and measurements</summary>

Viterbi seeks the most probable complete sequence of locations, rather than a separate summary at each time step.

The `additional_track_quantities` setting accepts only `"speed"` and `"distance"`. Afterward, MovingPandas can calculate acceleration (`add_acceleration`), direction (`add_direction`), changes in direction (`add_angular_difference`), and time intervals (`add_timedelta`) from the returned trajectories. These are separate processing steps; see the [MovingPandas trajectory API](https://movingpandas.readthedocs.io/en/main/api/trajectory.html).

</details>

In [ ]:
from pangeo_fish.helpers import compute_states

states = compute_states(
    target_root=f"{fit_root}",
    storage_options=storage_options,
    chunks=default_chunk_dims,
    save=True,
)

### Optional: export the posterior for Gridlook Preview

After checking the result, run the following in a new cell. Choose a new local path or an S3 URL in a bucket you can write to. This creates a separate viewing export with DGGS v1 metadata, one time step per chunk, and dense HEALPix parent tiles. Missing children are NaN; existing probabilities are unchanged.

```python
from pangeo_fish.storage import write_dggs_zarr

preview_path = f"{target_root}/states-preview.zarr"
write_dggs_zarr(states, preview_path, storage_options=storage_options)
```

For a remote export, use your write credentials in `storage_options`. Serve the store via HTTPS with read access and CORS for the viewer, then open `https://grid4earth.eu/gridlook-preview/#https://YOUR-ENDPOINT/YOUR-PREFIX/states-preview.zarr` and select `states`. See [the publishing guide](../docs/dggs-output.md) for chunk settings and HTTP checks. The exporter refuses to overwrite an existing store.

The probabilities in `states` should sum to approximately **1** at each time step. A departure from 1 means missing or invalid values at that time step.

In [ ]:
states["states"].sum(dims).hvplot(
    width=500, ylim=(0, 2), title="Total probability per time step"
)

In [ ]:
states = xr.open_dataset(
    f"{fit_root}/states.zarr",
    engine="zarr",
    chunks=default_chunk_dims,
    inline_array=True,
    storage_options=storage_options,
)

In [ ]:
states.healpix.plot("states")

## 11. Explore the maps and tracks

Compare the mean and mode tracks, and inspect the probability maps over time. Where is the fish's location tightly constrained, and where are several areas plausible?

The **emission** maps show support from the observations. The **state** maps also account for movement between time steps. Compare their patterns to see how the movement model affects the estimated locations.

### 11.1 Compare the tracks

Where do the mean and mode tracks agree or differ? Use the probability maps to interpret these differences.

In [ ]:
from pangeo_fish.helpers import decode_trajectories

trajectories = decode_trajectories(
    target_root=f"{fit_root}",
    storage_options=storage_options,
    chunks=default_chunk_dims,
    track_modes=track_modes,
    additional_track_quantities=additional_track_quantities,
    save=True,
)

In [ ]:
from pangeo_fish.helpers import plot_trajectories

plot_trajectories(
    target_root=f"{fit_root}",
    track_modes=track_modes,
    storage_options=storage_options,
    save_html=True,
)

### 11.2 View the probability distributions over time

In [ ]:
from pangeo_fish.helpers import open_distributions, render_distributions

In [ ]:
data = open_distributions(
    target_root=f"{fit_root}",
    storage_options=storage_options,
    chunks=default_chunk_dims,
    chunk_time=chunk_time,
)
data

The output above is a summary of the data available for plotting. Run the following cells to create an MP4 animation and play it directly below the code.

`time_step = 1` requests a frame at every data time step; `3` requests every third step. This changes only the video sampling. The current video tool omits the final time step, even when this setting is 1.

Watch for periods when the possible locations narrow or spread out, and compare these with the acoustic detection times.

In [ ]:
import numpy as np

lon = data["longitude"].values  # (nx,)
lat = data["latitude"].values  # (ny,)
lon_2d, lat_2d = np.meshgrid(
    lon, lat
)  # (ny, nx), two-dimensional geographic coordinates

data_2d = data.rename({"latitude": "y", "longitude": "x"})
data_2d = data_2d.assign_coords(
    longitude=(("y", "x"), lon_2d),
    latitude=(("y", "x"), lat_2d),
)
data_2d

In [ ]:
video_filename = render_distributions(
    data=data_2d,
    output_path=f"{fit_root}/states",
    xlim=bbox["longitude"],
    ylim=bbox["latitude"],
    time_step=time_step,
    extension="mp4",
    frames_dir="video",
    remove_frames=True,
    storage_options=storage_options,
)

In [ ]:
from IPython.display import Video, display

display(Video(filename=video_filename, embed=True))

In [ ]:
print("Your results are in:", target_root)

## Optional appendix: compare movement assumptions

The main task is complete. To explore the influence of movement assumptions, the following cells compare the fitted model with two chosen values of `sigma`. Look at how the probability maps and tracks change.

Each comparison uses a separate output folder. Keep the supplied settings for a first comparison; more advanced analyses can also use different movement parameters over different time intervals.

In [ ]:
import json
import warnings
from pathlib import Path
import fsspec
from pangeo_fish.helpers import create_parameters, hand_test_parameter
from pangeo_fish.helpers import optimize_pdf

First, copy the combined emission distribution into a separate output folder.

In [ ]:
from pangeo_fish.storage import prepare_dggs_zarr

# Open the distributions
emission = xr.open_dataset(
    f"{fit_root}/combined.zarr",
    engine="zarr",
    chunks=default_chunk_dims,
    inline_array=True,
    storage_options=storage_options,
)
# Create a new folder, using the current Dask chunks rather than source encoding.
for name in emission.variables:
    emission[name].encoding.pop("chunks", None)
prepare_dggs_zarr(emission).to_zarr(
    f"{fit_root}/hand_sigma0.1/combined.zarr",  # replace "hand_sigma" by another name
    mode="w",
    consolidated=True,
    storage_options=storage_options,
    zarr_format=2,
)

In [ ]:
emission

Save the chosen movement parameters in the same folder.

In [ ]:
params = create_parameters(
    sigma=[0.0003],
    sigma_indices=None,
    n_steps=emission.sizes["time"],
    single_behav=True,
    class_name="UpDownGaussian1DHealpix",  # Convolution class for the movement model.
    target_root=f"{fit_root}/hand_sigma0.1",  # Match the folder used above.
    save_parameters=True,
)
params

Alternatively, `test_parameter()` saves the combined emission distribution and the chosen parameters in a comparison folder. Pass an existing dataset as `emission`, or leave it as `None` to load `combined.zarr` from Section 6.

In [ ]:
params = hand_test_parameter(
    emission=None,
    sigma_tested=[0.03],
    sigma_indices=None,
    Conv_method="UpDownGaussian1DHealpix",
    target_root=fit_root,
    saving_root="/hand_sigma10",
    default_chunk_dims=default_chunk_dims,
    storage_options=storage_options,
)

In [ ]:
from pangeo_fish.helpers import compute_states

states_list = [
    compute_states(
        target_root=f"{fit_root}/{subfolder}",
        storage_options=storage_options,
        chunks=default_chunk_dims,
        save=True,
    )
    for subfolder in [
        "",
        "hand_sigma0.1",
        "hand_sigma10",
    ]
]

Compare the state distributions using synchronized maps, then compare the trajectories below.

In [ ]:
from pangeo_fish.helpers import multi_map_with_synced_sliders, multiplot_healpix

liste = multiplot_healpix(
    [
        (states_list[0], ["states"]),
        (states_list[1], ["states"]),
        (states_list[2], ["states"]),
    ],
)
liste

multi_map_with_synced_sliders(liste)

In [ ]:
from pangeo_fish.helpers import decode_trajectories

traj_list = [
    decode_trajectories(
        target_root=f"{fit_root}/{subfolder}",
        storage_options=storage_options,
        track_modes=comparison_track_modes,
        chunks=default_chunk_dims,
        save=True,
    )
    for subfolder in [
        "",
        "hand_sigma0.1",
        "hand_sigma10",
    ]
]

traj_plots = [
    plot_trajectories(
        target_root=f"{fit_root}/{subfolder}",
        track_modes=comparison_track_modes,
        storage_options=storage_options,
        save_html=True,
    ).options(title=f"Model's Folder: {subfolder}")
    for subfolder in [
        "",
        "hand_sigma0.1",
        "hand_sigma10",
    ]
]
(traj_plots[0] + traj_plots[1] + traj_plots[2]).cols(2)

## Appendix: information for bug reports

If something does not work, we need to know exactly which software you used. Run the next cell (it works at any time, even if earlier cells failed) and copy **all of its output** into your [issue](https://github.com/pangeo-fish/pangeo-fish-etn2026/issues) or e-mail, together with the error message.

The output answers three questions:

- **Which computing environment?** The Docker image your Jupyter server was started from (registry, name and tag, e.g. `ghcr.io/tinaok/pangeo-fish-notebook:2026.06.04-10751a5`). The tag ends with the Pangeo-Fish commit used to build it.
- **Which Pangeo-Fish code?** The Git repository and commit of the Pangeo-Fish source installed in that image.
- **Which notebook?** The commit of this repository, and whether you changed the notebook.
